# MASTER — Core Model Reproduction (Regression, Alpha158)

This notebook trains and evaluates **MASTER** (Market-Guided Stock Transformer, Li et al., AAAI 2024) on the DJI universe, as the first "core model" reproduction for the thesis *Applying Explainability Methods to Financial Models*.

**Design choices**
- All model/data-loading code below is **duplicated, not imported**, from `code/finbench/Regression/MASTER/` (`master.py`, `load_dataset.py`, `base_model.py`, `utils.py`, `train.py`). Nothing in `code/finbench/` is modified or called into — FinBench is treated as a read-only reference implementation.
- Data is read from `code/data/<universe>/` (already generated locally), not from `code/finbench/Evaluation/data/` (which the original `train.py` defaults to and which is not populated in this checkout).
- Sequence length / prediction length default to **T=20, L=5** (FinBench's own "weekly" lookback/horizon configuration for Alpha158 models), not MASTER's own demo defaults (`seq_len=8, pred_len=10`), so results stay comparable with the other core models later.
- Dates default to **FinBench's rolling phase 1**: train 2015–2018, validate 2019, test 2020.

**Known environment issue (found while building this notebook, not fixed here)**: `scipy` in `.venv` fails to import (`DLL load failed ... An Application Control policy has blocked this file`), which breaks `sklearn` and pandas' `method='spearman'` correlation. To keep this notebook runnable, IC/RankIC below are computed without scipy (Spearman = Pearson correlation of ranks) and R² is computed manually instead of via `sklearn.metrics.r2_score`. This is a workaround, not a fix — libraries used later for explainability (e.g. `shap`) will very likely need real scipy/sklearn, so the underlying Application Control block should be resolved (whitelist the venv, or reinstall scipy/sklearn via conda instead of pip) before that stage.

**Note on a bug found in the original code**: `finbench/Regression/MASTER/utils.py` calls `r2_score(preds_flat, labels_flat)`, i.e. predictions passed as `y_true` and labels as `y_pred` — backwards from the standard convention. This notebook computes R² with the correct `(y_true=labels, y_pred=preds)` order; R² values here may therefore differ slightly from numbers reproduced by running the original script unmodified.

In [21]:
import copy
import json
import math
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.modules.linear import Linear
from torch.nn.modules.dropout import Dropout
from torch.nn.modules.normalization import LayerNorm
from torch.utils.data import Dataset, Sampler, DataLoader
from tqdm.auto import tqdm

## 1. Configuration

In [19]:
# --- Paths -----------------------------------------------------------------
# This notebook lives in code/notebooks/. Data lives in code/data/, one level up,
# laid out as:
#   code/data/<universe>/<universe>_alpha158.csv
#   code/data/<universe>/<universe>.csv                 (OHLCV, for labels)
#   code/data/<universe>/<us|eu>_market.csv              (market-level gate features)
#   code/data/constituents/eodhd/<universe>.csv          (index membership dates)
DATA_ROOT = Path("..") / "data"
RESULTS_ROOT = Path("..") / "results"

# --- Universe / task ---------------------------------------------------------
UNIVERSE = "nasdaq100"  # 'dji', 'nasdaq100', 'sp500', 'sx5e', 'sxxp'
NATION = "us"  # 'us' for dji/nasdaq100/sp500, 'eu' for sx5e/sxxp -> selects <nation>_market.csv

# --- Sequence / horizon config ------------------------------------------------
# FinBench's rolling protocol pairs lookback T with horizon L as
# (T=5, L=1), (T=20, L=5), (T=60, L=20) for Alpha158-based models.
# Starting with the 'weekly' (T=20, L=5) configuration.
SEQ_LEN = 20
PRED_LEN = 5

# --- Rolling-phase dates ------------------------------------------------------
# FinBench rolling phase 1: train 2015-2018, valid 2019, test 2020.
START_DATE = "2015-01-01"
END_TRAIN_DATE = "2018-12-31"
START_VALID_DATE = "2019-01-01"
END_VALID_DATE = "2019-12-31"
START_TEST_DATE = "2020-01-01"
END_DATE = "2020-12-31"

# --- MASTER hyperparameters (as in the original MASTER train.py defaults) ----
D_MODEL = 256
T_NHEAD = 4
S_NHEAD = 2
DROPOUT = 0.5
BETA = 5
N_EPOCH = 40
LR = 1e-5
GATE_INPUT_START_INDEX = 157  # number of raw Alpha158 feature columns used as model input

# --- Model selection / early stopping -----------------------------------------
# NOTE: the original finbench/Regression/MASTER/train.py stops as soon as the raw
# TRAIN loss drops below a fixed threshold (0.95). Since labels are cross-sectionally
# z-scored to unit variance each day, an MSE of ~1.0 is roughly the 'no-skill' floor
# (always predicting the mean) -- 0.95 is barely below that floor, so that criterion
# can trigger after only a handful of epochs on noisy data (e.g. DJI's ~25-30 stocks
# per day) well before the model has actually converged, and it never looks at
# validation performance at all. We replace it with the standard approach: keep the
# checkpoint with the best validation RankIC seen so far, and stop early only if
# validation RankIC hasn't improved for PATIENCE consecutive epochs.
PATIENCE = 10  # epochs without validation RankIC improvement before stopping early

SEED = 42
NUM_WORKERS = 0  # kept at 0 to avoid Windows multiprocessing/DataLoader issues in notebooks

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_SAVE_PATH = RESULTS_ROOT / "Regression" / "MASTER" / UNIVERSE / f"sl{SEQ_LEN}_pl{PRED_LEN}"
METRICS_PATH = RESULTS_ROOT / "Regression" / "MASTER" / UNIVERSE / f"seed{SEED}" / f"y{START_TEST_DATE[:4]}"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)
METRICS_PATH.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

print(f"device={DEVICE}, universe={UNIVERSE}, seq_len={SEQ_LEN}, pred_len={PRED_LEN}")

device=cuda, universe=nasdaq100, seq_len=20, pred_len=5


## 2. Data loading and feature merge

Duplicated from `finbench/Regression/MASTER/train.py` and `utils.py`: filter index constituents by membership date, merge Alpha158 features with market-level "gate" features, attach the forward-return label, and restrict to tickers present during the training window.

In [3]:
def filter_constituents_by_date(constituents: pd.DataFrame, test_start_date: str) -> pd.DataFrame:
    """Keep only index constituents active on `test_start_date`.

    Duplicated from finbench/Regression/MASTER/utils.py (unmodified logic).
    """
    if not all(col in constituents.columns for col in ["StartDate", "EndDate"]):
        raise ValueError('The "constituents" DataFrame must contain StartDate and EndDate columns')

    start_dates = pd.to_datetime(constituents["StartDate"])
    end_dates = pd.to_datetime(constituents["EndDate"])
    test_start = pd.to_datetime(test_start_date)

    start_dates = start_dates.fillna(pd.Timestamp.min)
    end_dates = end_dates.fillna(pd.Timestamp.max)

    is_active = (start_dates < test_start) & (end_dates >= test_start)
    return constituents[is_active].copy()


def select_valid_ticker(df: pd.DataFrame, start_date: str, end_date: str) -> pd.DataFrame:
    """Keep only tickers that have at least one row in [start_date, end_date]."""
    df_train = df[(df["date"] >= start_date) & (df["date"] <= end_date)]
    tickers = df_train["instrument"].drop_duplicates().tolist()
    return df[df["instrument"].isin(tickers)]


def extract_labels(df: pd.DataFrame, universe: str, pred_len: int) -> pd.DataFrame:
    """Forward `pred_len`-step return on adjusted close, merged in as column 'Label'."""
    df_close = pd.read_csv(DATA_ROOT / universe / f"{universe}.csv")[["date", "instrument", "adj_close"]]
    df_close = df_close.sort_values(["instrument", "date"])
    df_close["Label"] = df_close.groupby("instrument")["adj_close"].transform(
        lambda x: (x.shift(-pred_len) - x) / x
    )
    return df.merge(df_close[["date", "instrument", "Label"]], on=["date", "instrument"], how="left")

In [4]:
print("Loading Alpha158 features (large file, may take a while)...")
df_alpha = pd.read_csv(DATA_ROOT / UNIVERSE / f"{UNIVERSE}_alpha158.csv")
shape_a = df_alpha.shape[1]

constituents = pd.read_csv(DATA_ROOT / "constituents" / "eodhd" / f"{UNIVERSE}.csv")
active_tickers = filter_constituents_by_date(constituents, START_TEST_DATE)
df_alpha = df_alpha[df_alpha["instrument"].isin(active_tickers["EODHD"].tolist())]

print("Loading market-level gate features...")
market_index = pd.read_csv(DATA_ROOT / UNIVERSE / f"{NATION}_market.csv")
shape_m = market_index.shape[1]
df_alpha = pd.merge(df_alpha, market_index, how="left", on="date")

# Matches finbench/Regression/MASTER/train.py's index arithmetic exactly:
# total feature columns (excluding date/instrument) after the merge, minus the
# Label column added by extract_labels below, gives the last raw-feature index
# before the market/gate feature block.
GATE_INPUT_END_INDEX = shape_m + shape_a - 3

df_alpha = extract_labels(df_alpha, UNIVERSE, PRED_LEN)
df_alpha = select_valid_ticker(df_alpha, START_DATE, END_TRAIN_DATE)

print(f"df_alpha shape: {df_alpha.shape}, gate_input_start_index={GATE_INPUT_START_INDEX}, "
      f"gate_input_end_index={GATE_INPUT_END_INDEX}")

Loading Alpha158 features (large file, may take a while)...
Loading market-level gate features...
df_alpha shape: (3013578, 223), gate_input_start_index=157, gate_input_end_index=220


## 3. Robust z-score normalization

Median/MAD normalization fit **only on the training window**, then applied identically to the train/valid/test splits below (avoids leaking validation/test statistics into training).

In [5]:
class RobustZScoreNormalization:
    """Median/MAD z-score, fit on the training window only, clipped to [-3, 3].

    Duplicated from finbench/Regression/MASTER/load_dataset.py.
    """

    def __init__(self, df: pd.DataFrame, eps: float = 1e-12):
        self.eps = eps
        self.z_score_cols = df.columns[2:-1]  # exclude 'date', 'instrument', 'Label'
        self.median = df.median(numeric_only=True)
        abs_dev = df[self.z_score_cols].transform(lambda x: (x - x.median()).abs())
        self.mad = abs_dev.median(numeric_only=True) + self.eps

    def robust_zscore(self, df: pd.DataFrame) -> pd.DataFrame:
        mad = self.mad * 1.4826  # scale MAD to match standard deviation
        df[self.z_score_cols] = (df[self.z_score_cols] - self.median[self.z_score_cols]) / mad[self.z_score_cols]
        df[self.z_score_cols] = np.clip(df[self.z_score_cols], -3, 3)
        return df


robust_z_score = RobustZScoreNormalization(
    df_alpha[(df_alpha["date"] >= START_DATE) & (df_alpha["date"] <= END_TRAIN_DATE)]
)

## 4. Dataset construction

Each sample corresponds to **one trading date**, holding a `[n_tickers, seq_len, n_features]` window of history for every ticker that has data up to that date. Validation/test windows are extended backward so the first prediction date still has a full lookback of history.

In [6]:
class CSVDataset(Dataset):
    """One sample = one trading date, holding a [n_tickers, seq_len, n_features] window.

    Duplicated from finbench/Regression/MASTER/load_dataset.py.
    """

    def __init__(self, df_alpha, seq_len, pred_len, start_date, end_date, z_score, period="train"):
        self.seq_len = seq_len
        self.pred_len = pred_len

        all_dates = df_alpha["date"].drop_duplicates().sort_values().reset_index(drop=True)

        if period in ("valid", "test"):
            # extend the window backward so the first prediction date still has
            # `seq_len + pred_len - 1` days of history available
            all_dates_dt, start_date_dt = pd.to_datetime(all_dates), pd.to_datetime(start_date)
            idx = (all_dates_dt >= start_date_dt).idxmax() - seq_len - pred_len + 1
            start_date = all_dates[idx]

        df_alpha = df_alpha[(df_alpha["date"] >= start_date) & (df_alpha["date"] <= end_date)].copy()
        df_alpha = df_alpha.groupby("instrument", group_keys=False).apply(lambda g: g.iloc[: -self.pred_len])
        df_alpha = df_alpha.reset_index(drop=True)
        df_alpha = df_alpha.sort_values(["instrument", "date"]).reset_index(drop=True)

        print(f"Applying z-score normalization for {period} period...")
        df_alpha = z_score.robust_zscore(df_alpha)
        df_alpha = df_alpha.fillna(0)

        self.input_dates, self.output_dates, all_dates = self.extract_dates(all_dates, start_date, end_date)

        self.feature_cols = [c for c in df_alpha.columns if c not in ["date", "instrument"]]

        self.ticker_dfs = {
            ticker: group.reset_index(drop=True)
            for ticker, group in df_alpha.groupby("instrument", group_keys=False)
            if len(group) >= seq_len
        }

        print(f"Extracting windows aligned by date for {period} period...")
        self.extract_windows_aligned_by_date(all_dates)
        print(f"Dataset loaded ({period}): {len(self.date_to_idx)} dates.")

    def extract_windows_aligned_by_date(self, dates):
        self.date_to_data = {}
        self.date_to_idx = []
        self.tickers_to_date = []

        for current_date in tqdm(dates):
            sequences, tickers = [], []
            for ticker, ticker_df in self.ticker_dfs.items():
                sub_df = ticker_df[ticker_df["date"] <= current_date]
                if len(sub_df) >= self.seq_len:
                    window = sub_df.iloc[-self.seq_len :]
                    if window["date"].iloc[-1] == current_date:
                        sequences.append(window[self.feature_cols].to_numpy())
                        tickers.append(ticker)

            if sequences:
                self.date_to_data[current_date] = np.stack(sequences, axis=0)  # [n_ticker, seq_len, feature_dim]
                self.date_to_idx.append(current_date)
                self.tickers_to_date.append(tickers)

    def extract_dates(self, dates, start_date, end_date):
        dates = dates[(dates >= start_date) & (dates <= end_date)].to_list()
        output_dates = dates[self.seq_len + self.pred_len - 1 :]
        input_dates = dates[self.seq_len - 1 : -self.pred_len]
        return input_dates, output_dates, dates

    def __getitem__(self, idx):
        date = self.date_to_idx[idx]
        return torch.tensor(self.date_to_data[date], dtype=torch.float32)

    def __len__(self):
        return len(self.date_to_idx)


class DailyBatchSampler(Sampler):
    """Each 'batch' is exactly one trading date (the dataset is already grouped by date).

    Duplicated from finbench/Regression/MASTER/load_dataset.py (DailyBatchSamplerRandomSP500),
    renamed since it is not SP500-specific.
    """

    def __init__(self, data_source, shuffle=False):
        super().__init__()
        self.data_source = data_source
        self.shuffle = shuffle

    def __iter__(self):
        indices = np.arange(len(self.data_source))
        if self.shuffle:
            np.random.shuffle(indices)
        return iter(indices)

    def __len__(self):
        return len(self.data_source)

In [7]:
dl_train = CSVDataset(df_alpha, SEQ_LEN, PRED_LEN, START_DATE, END_TRAIN_DATE, robust_z_score, period="train")
dl_valid = CSVDataset(df_alpha, SEQ_LEN, PRED_LEN, START_VALID_DATE, END_VALID_DATE, robust_z_score, period="valid")
dl_test = CSVDataset(df_alpha, SEQ_LEN, PRED_LEN, START_TEST_DATE, END_DATE, robust_z_score, period="test")

C:\Users\Luca Villani\AppData\Local\Temp\ipykernel_3356\1588494549.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_alpha = df_alpha.groupby("instrument", group_keys=False).apply(lambda g: g.iloc[: -self.pred_len])


Applying z-score normalization for train period...
Extracting windows aligned by date for train period...


  0%|          | 0/1006 [00:00<?, ?it/s]

Dataset loaded (train): 982 dates.


C:\Users\Luca Villani\AppData\Local\Temp\ipykernel_3356\1588494549.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_alpha = df_alpha.groupby("instrument", group_keys=False).apply(lambda g: g.iloc[: -self.pred_len])


...

Dataset loaded (valid): 252 dates.


C:\Users\Luca Villani\AppData\Local\Temp\ipykernel_3356\1588494549.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_alpha = df_alpha.groupby("instrument", group_keys=False).apply(lambda g: g.iloc[: -self.pred_len])


Applying z-score normalization for test period...
Extracting windows aligned by date for test period...


  0%|          | 0/277 [00:00<?, ?it/s]

Dataset loaded (test): 253 dates.


## 5. Model architecture — MASTER

Duplicated unmodified from `finbench/Regression/MASTER/master.py`: a market-guided feature gate, intra-stock temporal attention (`TAttention`), inter-stock cross-sectional attention (`SAttention`), and a query-based temporal pooling head (`TemporalAttention`). Each attention module computes its softmax weights internally; a later notebook will patch these classes to also *return* those weights for the explainability analysis — not done here to keep this first pass a faithful reproduction.

In [8]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[: x.shape[1], :]


class SAttention(nn.Module):
    """Inter-stock (cross-sectional) multi-head self-attention."""

    def __init__(self, d_model, nhead, dropout):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.temperature = math.sqrt(self.d_model / nhead)

        self.qtrans = nn.Linear(d_model, d_model, bias=False)
        self.ktrans = nn.Linear(d_model, d_model, bias=False)
        self.vtrans = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.ModuleList([Dropout(p=dropout) for _ in range(nhead)])

        self.norm1 = LayerNorm(d_model, eps=1e-5)
        self.norm2 = LayerNorm(d_model, eps=1e-5)
        self.ffn = nn.Sequential(
            Linear(d_model, d_model), nn.ReLU(), Dropout(p=dropout),
            Linear(d_model, d_model), Dropout(p=dropout),
        )

    def forward(self, x):
        x = self.norm1(x)
        q = self.qtrans(x).transpose(0, 1)
        k = self.ktrans(x).transpose(0, 1)
        v = self.vtrans(x).transpose(0, 1)

        dim = int(self.d_model / self.nhead)
        att_output = []
        for i in range(self.nhead):
            sl = slice(i * dim, None) if i == self.nhead - 1 else slice(i * dim, (i + 1) * dim)
            qh, kh, vh = q[:, :, sl], k[:, :, sl], v[:, :, sl]
            atten = torch.softmax(torch.matmul(qh, kh.transpose(1, 2)) / self.temperature, dim=-1)
            atten = self.attn_dropout[i](atten)
            att_output.append(torch.matmul(atten, vh).transpose(0, 1))
        att_output = torch.concat(att_output, dim=-1)

        xt = x + att_output
        xt = self.norm2(xt)
        return xt + self.ffn(xt)


class TAttention(nn.Module):
    """Intra-stock (temporal) multi-head self-attention."""

    def __init__(self, d_model, nhead, dropout):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.qtrans = nn.Linear(d_model, d_model, bias=False)
        self.ktrans = nn.Linear(d_model, d_model, bias=False)
        self.vtrans = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.ModuleList([Dropout(p=dropout) for _ in range(nhead)]) if dropout > 0 else []

        self.norm1 = LayerNorm(d_model, eps=1e-5)
        self.norm2 = LayerNorm(d_model, eps=1e-5)
        self.ffn = nn.Sequential(
            Linear(d_model, d_model), nn.ReLU(), Dropout(p=dropout),
            Linear(d_model, d_model), Dropout(p=dropout),
        )

    def forward(self, x):
        x = self.norm1(x)
        q, k, v = self.qtrans(x), self.ktrans(x), self.vtrans(x)

        dim = int(self.d_model / self.nhead)
        att_output = []
        for i in range(self.nhead):
            sl = slice(i * dim, None) if i == self.nhead - 1 else slice(i * dim, (i + 1) * dim)
            qh, kh, vh = q[:, :, sl], k[:, :, sl], v[:, :, sl]
            atten = torch.softmax(torch.matmul(qh, kh.transpose(1, 2)), dim=-1)
            if self.attn_dropout:
                atten = self.attn_dropout[i](atten)
            att_output.append(torch.matmul(atten, vh))
        att_output = torch.concat(att_output, dim=-1)

        xt = x + att_output
        xt = self.norm2(xt)
        return xt + self.ffn(xt)


class Gate(nn.Module):
    """Market-status-conditioned per-feature reweighting of the raw Alpha inputs."""

    def __init__(self, d_input, d_output, beta=1.0):
        super().__init__()
        self.trans = nn.Linear(d_input, d_output)
        self.d_output = d_output
        self.t = beta

    def forward(self, gate_input):
        output = self.trans(gate_input)
        output = torch.softmax(output / self.t, dim=-1)
        return self.d_output * output


class TemporalAttention(nn.Module):
    """Query-based pooling over the time dimension using the last timestep as query."""

    def __init__(self, d_model):
        super().__init__()
        self.trans = nn.Linear(d_model, d_model, bias=False)

    def forward(self, z):
        h = self.trans(z)  # [N, T, D]
        query = h[:, -1, :].unsqueeze(-1)
        lam = torch.matmul(h, query).squeeze(-1)  # [N, T]
        lam = torch.softmax(lam, dim=1).unsqueeze(1)
        return torch.matmul(lam, z).squeeze(1)  # [N, D]


class MASTER(nn.Module):
    """MASTER: Market-Guided Stock Transformer (Li et al., AAAI 2024).

    Duplicated from finbench/Regression/MASTER/master.py, unmodified architecture.
    """

    def __init__(self, d_feat, d_model, t_nhead, s_nhead, T_dropout_rate, S_dropout_rate,
                 gate_input_start_index, gate_input_end_index, beta):
        super().__init__()
        self.gate_input_start_index = gate_input_start_index
        self.gate_input_end_index = gate_input_end_index
        self.d_gate_input = gate_input_end_index - gate_input_start_index
        self.feature_gate = Gate(self.d_gate_input, d_feat, beta=beta)

        self.layers = nn.Sequential(
            nn.Linear(d_feat, d_model),
            PositionalEncoding(d_model),
            TAttention(d_model=d_model, nhead=t_nhead, dropout=T_dropout_rate),
            SAttention(d_model=d_model, nhead=s_nhead, dropout=S_dropout_rate),
            TemporalAttention(d_model=d_model),
            nn.Linear(d_model, 1),
        )

    def forward(self, x):
        src = x[:, :, : self.gate_input_start_index]  # [N, T, D]
        gate_input = x[:, -1, self.gate_input_start_index : self.gate_input_end_index]
        src = src * torch.unsqueeze(self.feature_gate(gate_input), dim=1)
        return self.layers(src).squeeze(-1)

## 6. Training utilities

Duplicated from `finbench/Regression/MASTER/base_model.py`, with two documented deviations: Spearman rank correlation and R² are computed without scipy/sklearn (see the environment note in section 1), and R² uses the standard `(y_true, y_pred)` argument order.

In [9]:
def zscore(x: torch.Tensor) -> torch.Tensor:
    return (x - x.mean()).div(x.std())


def drop_extreme(x: torch.Tensor):
    """Drop the most extreme 2.5% of labels on each tail (cross-sectional outlier removal)."""
    sorted_tensor, indices = x.sort()
    n = x.shape[0]
    percent_2_5 = int(0.025 * n)
    if percent_2_5 != 0:
        filtered_indices = indices[percent_2_5:-percent_2_5]
        mask = torch.zeros_like(x, device=x.device, dtype=torch.bool)
        mask[filtered_indices] = True
        return mask, x[mask]
    return torch.ones_like(x, device=x.device, dtype=torch.bool), x


def calc_ic(pred: np.ndarray, label: np.ndarray):
    """Daily cross-sectional IC (Pearson) and Rank IC (Spearman via rank + Pearson,
    avoiding the scipy dependency that is currently broken in this environment)."""
    df = pd.DataFrame({"pred": pred, "label": label})
    ic = df["pred"].corr(df["label"])
    ric = df["pred"].rank().corr(df["label"].rank())
    return ic, ric


def r2_score_manual(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true, y_pred = np.asarray(y_true).ravel(), np.asarray(y_pred).ravel()
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return float(1 - ss_res / ss_tot)


def loss_fn(pred: torch.Tensor, label: torch.Tensor) -> torch.Tensor:
    mask = ~torch.isnan(label)
    return torch.mean((pred[mask] - label[mask]) ** 2)


def make_loader(dataset, shuffle, drop_last):
    sampler = DailyBatchSampler(dataset, shuffle=shuffle)
    return DataLoader(dataset, sampler=sampler, drop_last=drop_last, num_workers=NUM_WORKERS, pin_memory=True)


def train_epoch(model, optimizer, data_loader):
    model.train()
    losses = []
    for data in data_loader:
        data = torch.squeeze(data, dim=0)  # [N, T, F]; F = alpha + market features + Label
        feature = data[:, :, 0:-1].to(DEVICE)
        label = data[:, -1, -1].to(DEVICE)

        mask, label = drop_extreme(label)
        feature = feature[mask, :, :]
        label = zscore(label)

        pred = model(feature.float())
        loss = loss_fn(pred, label)
        losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_value_(model.parameters(), 3.0)
        optimizer.step()

    return float(np.mean(losses))


def predict(model, data_loader):
    model.eval()
    preds, labels, ic, ric = [], [], [], []

    for data in data_loader:
        data = torch.squeeze(data, dim=0)
        feature = data[:, :, 0:-1].to(DEVICE)
        label = zscore(data[:, -1, -1])

        with torch.no_grad():
            pred = model(feature.float()).detach().cpu().numpy()

        daily_ic, daily_ric = calc_ic(pred, label.detach().numpy())
        ic.append(daily_ic)
        ric.append(daily_ric)
        preds.append(pred[:, np.newaxis])
        labels.append(label.detach().cpu().numpy()[:, np.newaxis])

    preds_flat = np.concatenate(preds, axis=0)
    labels_flat = np.concatenate(labels, axis=0)

    metrics = {
        "MSE": float(np.mean((preds_flat - labels_flat) ** 2)),
        "MAE": float(np.mean(np.abs(preds_flat - labels_flat))),
        "RMSE": float(np.sqrt(np.mean((preds_flat - labels_flat) ** 2))),
        "R2": r2_score_manual(labels_flat, preds_flat),
        "IC": float(np.nanmean(ic)),
        "RankIC": float(np.nanmean(ric)),
    }
    return preds, labels, metrics

## 7. Train

Trains for up to `N_EPOCH` epochs, evaluating on the validation split after every epoch. The checkpoint with the **best validation RankIC** is kept (not simply the last epoch's weights), and training stops early if validation RankIC hasn't improved for `PATIENCE` consecutive epochs. See the config-cell note in section 1 for why this replaces the original train-loss-threshold criterion.

In [18]:
train_loader = make_loader(dl_train, shuffle=True, drop_last=True)

model = MASTER(
    d_feat=GATE_INPUT_START_INDEX, d_model=D_MODEL, t_nhead=T_NHEAD, s_nhead=S_NHEAD,
    T_dropout_rate=DROPOUT, S_dropout_rate=DROPOUT,
    gate_input_start_index=GATE_INPUT_START_INDEX, gate_input_end_index=GATE_INPUT_END_INDEX,
    beta=BETA,
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = []
best_rankic = -float("inf")
best_epoch = -1
best_state = None
epochs_since_improve = 0

start = time.time()
for epoch in range(N_EPOCH):
    train_loss = train_epoch(model, optimizer, train_loader)

    valid_loader = make_loader(dl_valid, shuffle=False, drop_last=False)
    _, _, valid_metrics = predict(model, valid_loader)
    history.append({"epoch": epoch, "train_loss": train_loss, **valid_metrics})

    improved = valid_metrics["RankIC"] > best_rankic
    if improved:
        best_rankic = valid_metrics["RankIC"]
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_since_improve = 0
    else:
        epochs_since_improve += 1

    print(f"Epoch {epoch:02d} | train_loss {train_loss:.6f} | "
          f"valid MSE {valid_metrics['MSE']:.4f} MAE {valid_metrics['MAE']:.4f} "
          f"IC {valid_metrics['IC']:.4f} RankIC {valid_metrics['RankIC']:.4f}"
          f"{' (best)' if improved else ''}")

    if epochs_since_improve >= PATIENCE:
        print(f"No validation RankIC improvement for {PATIENCE} epochs, stopping early "
              f"(best epoch: {best_epoch}, RankIC: {best_rankic:.4f}).")
        break

print(f"Training time: {time.time() - start:.1f}s")

model.load_state_dict(best_state)
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)  # re-create defensively in case it was removed mid-run
torch.save(best_state, MODEL_SAVE_PATH / "model.pth")
print(f"Restored and saved best checkpoint (epoch {best_epoch}, valid RankIC {best_rankic:.4f}) "
      f"to {MODEL_SAVE_PATH / 'model.pth'}")

Epoch 00 | train_loss 1.094913 | valid MSE 1.0120 MAE 0.7158 IC -0.0027 RankIC -0.0034 (best)
Epoch 01 | train_loss 1.023645 | valid MSE 1.0069 MAE 0.7136 IC -0.0092 RankIC -0.0119
Epoch 02 | train_loss 1.007854 | valid MSE 1.0040 MAE 0.7127 IC 0.0057 RankIC 0.0035 (best)
Epoch 03 | train_loss 1.001801 | valid MSE 1.0029 MAE 0.7121 IC -0.0005 RankIC -0.0007
Epoch 04 | train_loss 0.998249 | valid MSE 1.0028 MAE 0.7123 IC 0.0052 RankIC 0.0039 (best)
Epoch 05 | train_loss 0.995687 | valid MSE 1.0074 MAE 0.7135 IC -0.0236 RankIC -0.0235
Epoch 06 | train_loss 0.993593 | valid MSE 1.0044 MAE 0.7131 IC -0.0028 RankIC -0.0021
Epoch 07 | train_loss 0.991606 | valid MSE 1.0055 MAE 0.7128 IC -0.0122 RankIC -0.0102
Epoch 08 | train_loss 0.990389 | valid MSE 1.0064 MAE 0.7137 IC -0.0118 RankIC -0.0113
Epoch 09 | train_loss 0.988388 | valid MSE 1.0073 MAE 0.7141 IC -0.0116 RankIC -0.0098
Epoch 10 | train_loss 0.986768 | valid MSE 1.0077 MAE 0.7143 IC -0.0084 RankIC -0.0068
Epoch 11 | train_loss 0.98

## 8. Evaluate on the test split and save results

In [11]:
test_loader = make_loader(dl_test, shuffle=False, drop_last=False)
preds, labels, test_metrics = predict(model, test_loader)
print("Test metrics:", test_metrics)

results = {
    "metrics": test_metrics,
    "preds": preds,
    "labels": labels,
    "pred_date": dl_test.output_dates,
    "last_date": dl_test.input_dates,
    "tickers": dl_test.tickers_to_date,
    "config": {
        "universe": UNIVERSE, "seq_len": SEQ_LEN, "pred_len": PRED_LEN, "seed": SEED,
        "start_test_date": START_TEST_DATE, "end_date": END_DATE,
        "best_epoch": best_epoch, "best_valid_rankic": best_rankic, "patience": PATIENCE,
    },
}

METRICS_PATH.mkdir(parents=True, exist_ok=True)  # re-create defensively in case it was removed mid-run
with open(METRICS_PATH / f"results_sl{SEQ_LEN}_pl{PRED_LEN}.pkl", "wb") as f:
    pickle.dump(results, f)
with open(METRICS_PATH / f"metrics_sl{SEQ_LEN}_pl{PRED_LEN}.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

pd.DataFrame(history).to_csv(METRICS_PATH / f"train_history_sl{SEQ_LEN}_pl{PRED_LEN}.csv", index=False)
print(f"Saved results to {METRICS_PATH}")

Test metrics: {'MSE': 1.0058350563049316, 'MAE': 0.7360533475875854, 'RMSE': 1.002913236618042, 'R2': -0.007862091064453125, 'IC': 0.006453476449829948, 'RankIC': 0.004076991454307433}
Saved results to ..\results\Regression\MASTER\sp500\seed42\y2020


## 9. Next steps

- Re-run with `UNIVERSE = "nasdaq100"` / `"sx5e"` and other `SEED` values to build the multi-universe, multi-seed grid needed for the thesis's cross-model comparison.
- Duplicate `FactorVAE` and `HIST` into their own notebooks the same way, keeping `SEQ_LEN`/`PRED_LEN`/date-range conventions consistent so results are directly comparable.
- Add a small patch to the `SAttention`/`TAttention`/`TemporalAttention` `forward()` methods above so they also return their softmax attention tensors (currently computed but discarded) — needed for the attention-inspection part of the explainability analysis, kept out of this first reproduction pass on purpose.
- Resolve the scipy/sklearn Application Control DLL block in `.venv` before starting the SHAP/LIME work, since the `shap` package depends on scipy internally.